<a href="https://colab.research.google.com/github/winicius87/NBA-2026-Prediction-with-Tensorflow/blob/main/NBA_2026_Prediction_with_Tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''''import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize pixel values (0–255) to (0–1)
x_train, x_test = x_train / 255.0, x_test / 255.0
model = keras.Sequential([
      layers.Flatten(input_shape=(28, 28)),   # Converts 2D image to 1D
          layers.Dense(128, activation='relu'),   # Hidden layer
              layers.Dropout(0.2),                    # Prevent overfitting
                  layers.Dense(10, activation='softmax')  # Output layer
                  ])
model.compile(
      optimizer='adam',
          loss='sparse_categorical_crossentropy',
              metrics=['accuracy']
              )

model.fit(x_train, y_train, epochs=5)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

# Save model
model.save('my_first_nn.h5')

# Load model
loaded_model = keras.models.load_model('my_first_nn.h5')


Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9104 - loss: 0.3034
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9563 - loss: 0.1472
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9664 - loss: 0.1109
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9723 - loss: 0.0905
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9767 - loss: 0.0742
313/313 - 1s - 2ms/step - accuracy: 0.9757 - loss: 0.0785

Test accuracy: 0.9757000207901001


In [ ]:
import tensorflow as tf
import numpy as np

# 1. Generate sample 3D data points lying on a plane (e.g., z = 2x - 3y + 5)
np.random.seed(42)
x_data = np.random.uniform(-10, 10, 100)
y_data = np.random.uniform(-10, 10, 100)
# Add some noise to the data
z_data = 2 * x_data - 3 * y_data + 5 + np.random.normal(0, 0.1, 100)

# 2. Combine X and Y into a single input matrix (Shape: 100 samples, 2 features)
xy_data = np.column_stack((x_data, y_data))

# 3. Define the single-layer neural network (no activation = linear model)
# Dense(1) inherently represents: Output = (Weight_X * X) + (Weight_Y * Y) + Bias
model = tf.keras.Sequential([
    tf.keras.layers.Dense(units=1, input_shape=(2,))
    ])

# 4. Compile model with Mean Squared Error loss and an optimizer
model.compile(optimizer='adam', loss='mean_squared_error')

# 5. Train the model to fit the plane
model.fit(xy_data, z_data, epochs=5000, verbose=0)

# 6. Extract the learned parameters
weights, bias = model.layers[0].get_weights()
a, b = weights[0][0], weights[1][0]
c = bias[0]

print(f"Learned Plane Equation: z = {a:.4f}x + {b:.4f}y + {c:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Learned Plane Equation: z = 1.9978x + -2.9949y + 5.0048


In [12]:
from google.colab import auth
import gspread
from google.auth import default

!google-drive-ocamlfuse -cc
# 1. Authenticate your Google Account
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SPREADSHEET_ID = "1Hz7oxYkP3dasB7IaPyubC2J3MKGGka1-l4UZ9hX-FcE"
spreadsheet = gc.open_by_key(SPREADSHEET_ID)
#spreadsheet = gc.open_by_url("https://docs.google.com/spreadsheets/d/1Hz7oxYkP3dasB7IaPyubC2J3MKGGka1-l4UZ9hX-FcE/")
#
# 2. Access your specific worksheet
#worksheet = sheet.get_worksheet(0) # 0 is the first tab
# 2. Open the Google Sheets file by its exact name
#spreadsheet = gc.open("Sasvsknicks")

# 3. Open the second sheet (tab)
# Note: get_worksheet(1) uses a 0-based index (0=first sheet, 1=second sheet)
worksheet = spreadsheet.get_worksheet(7) # Renamed to worksheet for clarity

# 4. Extract data and convert to TensorFlow compatible format (e.g., Pandas DataFrame)
import pandas as pd
raw_data = worksheet.get_all_values() # Get all data as list of lists

# Assuming the first row is the header
headers = raw_data[0]
data_rows = raw_data[1:]

# Generate unique column names to handle potential duplicates in raw_data[0]
seen_headers = {}
unique_headers = []
for h in headers:
    original_h = h
    counter = 1
    while h in seen_headers:
        h = f"{original_h}_{counter}"
        counter += 1
    seen_headers[h] = True
    unique_headers.append(h)

df_pandas = pd.DataFrame(data_rows, columns=unique_headers)

qcol = len(unique_headers)

print(headers[:qcol])

df_pandas = df_pandas.iloc[:, :qcol]  # Remaining columns are your features

# Convert the target column (first column) to numerical (0 or 1)
# Assuming 'W' maps to 1 and 'L' maps to 0
df_pandas.iloc[:, 1] = df_pandas.iloc[:, 1].apply(lambda x: 1 if str(x).strip().upper() == 'W' else (0 if str(x).strip().upper() == 'L' else pd.NA))

# Convert the rest of the columns (features) to numeric
for col in df_pandas.columns[2:]:
    df_pandas[col] = pd.to_numeric(df_pandas[col], errors='coerce')

# Drop any rows that resulted in NaN after conversion (e.g., empty cells, non-numeric strings in numeric columns)
df_pandas.dropna(inplace=True)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load data - already loaded into df_pandas
#
#choose the quarter cuts the table at that colum
#q1 col 19
#q2 col 19+18=37
#q3 col 37+18=55
#q4 col 55+18=73
n_predict = 1

last_row_first_column = raw_data[-1][0]

print(f"Columns  : {qcol } {last_row_first_column}")

#remove first n rows outliers
rem_n =0
# 2. Separate Target (1st column) and Features
y = df_pandas.iloc[rem_n :-n_predict, 1].values.astype(int)  # First column is your binary outcome (0 or 1)
X = df_pandas.iloc[rem_n :-n_predict, 2:qcol].values  # Remaining columns are your features

pred_y= df_pandas.iloc[-n_predict:, 1].values  # Remaining columns are your features

pred = df_pandas.iloc[-n_predict:, 2:qcol].values  # Remaining columns are your features


# 3. Scale the features (Neural networks perform best on scaled data)
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
pred_scaled = scaler.transform(pred)

#X_scaled = X_scaled0.iloc[:-n_predict]
#X_predict = X_scaled0.iloc[-n_predict:]


# 4. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, shuffle=True, test_size=0.1, random_state=42)


# Shared dense layer to extract latent team strengths


import tensorflow as tf
from tensorflow.keras import layers, Sequential

# Define the number of features
num_features = X.shape[1]
print(f"num_features {num_features }")
model = Sequential([
    layers.Dense(num_features   , activation='relu', input_shape=(num_features,)),
     layers.Dropout(0.3),  # Drop nodes to improve generalization
      #layers.Dense(16 , activation='relu'),
       layers.Dense(64 , activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        layers.Dropout(0.3), # Drop nodes to improve generalization
            layers.Dense(1, activation='sigmoid')  # 1 output neuron with Sigmoid activation for binary [0, 1] outcome
            ])
model.compile(
optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
#optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy']
)


callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
]


# Train using EarlyStopping to halt when validation loss stops improving
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)



# Train the model
history = model.fit(X_train, y_train,
                    epochs=150,
                    batch_size=32,
                    callbacks=[early_stop],
                    validation_split=0.2)

# Evaluate on test data
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_accuracy:.2f}")

# Predict probabilities (e.g., 0.85 chance of being Class 1)
y_pred_probs = model.predict(X_test)
# Convert probabilities to binary 0 or 1
y_pred_classes = (y_pred_probs >= 0.5).astype(int)
print(f"Test Predictions : {y_pred_classes}")


pred_probs = model.predict(pred_scaled)
pred_classes = (pred_probs >= 0.5).astype(int)
last_row_first_column = raw_data[-1][0]

#print(f"Two Predictions : {pred}")

print(f"Predictions : {pred_probs} ")

print(f"Columns  : {qcol } {last_row_first_column}")

print(f"Test Predictions : {pred_classes}")

/bin/bash: line 1: google-drive-ocamlfuse: command not found
['MATCH UP', 'W/L', 'MIN', 'PTS', 'FGM', 'FGA', 'FG%', '3:00 PM', '3PA', '3P%', 'FTM', 'FTA', 'FT%', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'PF', '#ERROR!']
Columns  : 22 Jun 03, 2026 - NYK @ SAS
num_features 20
Epoch 1/150


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.5556 - loss: 1.0726 - val_accuracy: 0.0000e+00 - val_loss: 1.3914
Epoch 2/150
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.2222 - loss: 1.2247 - val_accuracy: 0.0000e+00 - val_loss: 1.3547
Epoch 3/150
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.4444 - loss: 1.0041 - val_accuracy: 0.0000e+00 - val_loss: 1.3203
Epoch 4/150
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.5556 - loss: 1.1254 - val_accuracy: 0.0000e+00 - val_loss: 1.2869
Epoch 5/150
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.6667 - loss: 0.9731 - val_accuracy: 0.0000e+00 - val_loss: 1.2550
Epoch 6/150
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.2222 - loss: 1.0640 - val_accuracy: 0.0000e+00 - val_loss: 1.2236
Epoch 7/150
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.4444 - loss: 1.0412 - val_accuracy: 0.3333 - val_loss: 1.1924
Epoch 8/150
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.7778 - loss: 0.8633 - val_accuracy: 0.3333 -

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

# 1. Get raw model probabilities once
#raw_probs = model.predict(X_test)

# 2. Define the thresholds you want to test
thresholds = [0.5, 0.8]

for t in thresholds:
    print(f"\n" + "="*40)
    print(f" Classification Report for Threshold: {t} ")
    print("="*40)

    # Apply current threshold
    preds = (y_pred_probs >= t).astype(int)

    # Print metrics
    print(classification_report(y_test, preds, target_names=['L', 'W']))


 Classification Report for Threshold: 0.5 
              precision    recall  f1-score   support

           L       0.00      0.00      0.00         1
           W       0.67      1.00      0.80         2

    accuracy                           0.67         3
   macro avg       0.33      0.50      0.40         3
weighted avg       0.44      0.67      0.53         3


 Classification Report for Threshold: 0.8 
              precision    recall  f1-score   support

           L       0.00      0.00      0.00         1
           W       0.67      1.00      0.80         2

    accuracy                           0.67         3
   macro avg       0.33      0.50      0.40         3
weighted avg       0.44      0.67      0.53         3



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [ ]:
pred2  = [-0.6128214439,	55,	26.07005,	-3.0091,	29.07915,	18,	10, 21, 5]
# Reshape to a 2D array (1 sample, 3 features)
pred_2d = np.array(pred2 ).reshape(1, -1)


print(f"Pred : {pred} \n Scaled {pred_scaled}")
pred_scaled = scaler.transform(pred)
print(f"y scaled {pred_scaled }")

print(f"Pred : {pred_2d }")
pred2_scaled = scaler.transform(pred_2d )
print(f"y scaled {pred2_scaled }")


pred2_probs = model.predict(pred2_scaled)

print(f"Predictions : {pred2_probs} ")




#print(f"x scaled {X_scaled }")


red = [[-0.61282144,
        55.,
        26.07005,
        -3.0091,
        29.07915,
        18.,
        10.
        ]]



#pred = df_pandas.iloc[-n_predict:, 2:qcol].values  # Remaining columns are your features
# 3 rows (samples), 2 columns (features)
#pred = np.array([
#    [10, 0.5],
#    [20, 1.2],
#    [30, 0.9]
#])
#pred = np.array(flat_list).reshape(-1, 1)
#pred_scaled = scaler.fit_transform(pred)


#pred_probs = model.pre


#pred = df_pandas.iloc[-n_predict:, 2:qcol].values  # Remaining columns are your features
# 3 rows (samples), 2 columns (features)
#pred = np.array([
#    [10, 0.5],
#    [20, 1.2],
#    [30, 0.9]
#])
#pred = np.array(flat_list).reshape(-1, 1)
#pred_scaled = scaler.fit_transform(pred)


#pred_probs = model.predict(pred_scaled)

Pred : [[-0.51043723 56.         21.          7.         10.          4.
  28.4829     -7.4932    ]] 
 Scaled [[-0.52584987 -0.52584987 -0.31569579 -0.57902952 -2.26884112 -1.04474697
  -0.69220433 -1.5972271 ]]
y scaled [[-0.52584987 -0.52584987 -0.31569579 -0.57902952 -2.26884112 -1.04474697
  -0.69220433 -1.5972271 ]]
Pred : [[-0.61282144 55.         26.07005    -3.0091     29.07915    18.
  10.         21.          5.        ]]


ValueError: X has 9 features, but StandardScaler is expecting 8 features as input.

In [ ]:
import os
import random

# 1. Set python and numpy seeds
os.environ['PYTHONHASHSEED'] = '0'
random.seed(42)
np.random.seed(42)

# 2. Set TensorFlow seed
tf.random.set_seed(42)

# 3. Force CPU deterministic behavior (Optional, slows down GPUs)
os.environ['TF_DETERMINISTIC_OPS'] = '1'